<a href="https://colab.research.google.com/github/chanseck/Data-Analysis-with-Open-Source/blob/main/%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_4%EA%B0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# 오픈소스 기반 데이터 분석 4강 - 데이터 수집


## 4-1 CSV 파일 읽기

In [3]:
import pandas as pd

## data.csv 파일 읽기
df = pd.read_csv('data.csv', encoding='utf-8', sep=',', header=0,
                 index_col=None, skiprows=None, nrows=None)

print(df)

           날짜    체중  골격근량  체지방량
0  2025.02.06  64.7  30.0  11.1
1  2025.02.04  64.0  29.3  11.6


## 4-2 JSON 파일 읽기



In [5]:
import json
import pandas as pd

## data.json 파일 출력
with open('data.json', mode='r', encoding='utf-8') as f:
  data = json.load(f)
print(data)

## data.json 파일 DataFrame 읽기
df = pd.read_json('data.json', orient='records', encoding='utf-8')

print(df)

{'매출데이터': [{'월': '2025-01', '매출액': 1000000, '비용': 700000, '이익': 300000}, {'월': '2025-02', '매출액': 1200000, '비용': 800000, '이익': 400000}, {'월': '2025-03', '매출액': 1500000, '비용': 900000, '이익': 600000}]}
                                               매출데이터
0  {'월': '2025-01', '매출액': 1000000, '비용': 700000,...
1  {'월': '2025-02', '매출액': 1200000, '비용': 800000,...
2  {'월': '2025-03', '매출액': 1500000, '비용': 900000,...


## 4-3 텍스트 파일 읽기 및 데이터 추출

In [6]:
import re

## 파일(callcenter20250301.log) 오픈 및 읽기
with open('callcenter20250301.log', 'r', encoding='utf-8') as f:
    content = f.read()
## 주민등록번호 패턴 생성
pattern = re.compile(r'(\d{6})-(\d{7})')

## 주민등록번호 마스킹
masked_content = pattern.sub(r'\1-*******', content)

## 마스킹된 파일(callcenter20250301_masked.log) 오픈 및 쓰기
with open('callcenter20250301_masked.log', mode='w') as f:
  f.write(masked_content)

print("주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.")

주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.


## 4-4 Open-Meteo의 무료 날씨 API를 통한 특정 지역 온도 조회

In [7]:
import requests
import json

url = "https://api.open-meteo.com/v1/forecast?=&=&current=temperature_2m"
params = {
    "latitude": "37.58638333",
    "longitude": "127.0203333",
    "current": "temperature_2m"
}

try:
    ## URL 및 파라미터 전송
    response = requests.get(url, params=params)
    response.raise_for_status()

    ## JSON 데이터 읽기
    data = response.json()

    print("API 응답:", data)
    print("서울시 종로구의 현재 온도는 : {0}{1} 입니다.".format(data['current']['temperature_2m'], data['current_units']['temperature_2m']))

except requests.exceptions.RequestException as e:
    print(f"API 호출 실패: {e}")
except json.JSONDecodeError as e:
    print(f"JSON 파싱 실패: {e}")

API 응답: {'latitude': 37.6, 'longitude': 127.0, 'generationtime_ms': 1.9439458847045898, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 29.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C'}, 'current': {'time': '2026-10-09T07:00', 'interval': 900, 'temperature_2m': 23.1}}
서울시 종로구의 현재 온도는 : 23.1°C 입니다.


## 4-5 Selenium과 lxml을 이용한 웹 스크래핑

In [8]:
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install ./google-chrome-stable_current_amd64.deb -y
!pip install selenium webdriver_manager

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  136M  100  136M    0     0   333M      0 --:--:-- --:--:-- --:--:--  333M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 2 not upgraded.
Need to get 331 kB/144 MB of archives.
After this operation, 463 MB of addition

In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from lxml import html
import time

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')               # 브라우저 창 없이 실행
chrome_options.add_argument('--no-sandbox')             # 보안모드 비활성화 (Colab 필수)
chrome_options.add_argument('--disable-dev-shm-usage')  # 메모리 부족 방지 (Colab 필수)
chrome_options.add_argument('--window-size=1920x1080')  # 창 크기 설정(가상)
chrome_options.add_argument('--disable-gpu')            # GPU 가속 비활성화 (일부 환경 안정성)
chrome_options.binary_location = "/usr/bin/google-chrome-stable"  # Colab용 크롬 경로 지정

## 드라이버 실행
driver = webdriver.Chrome(options=chrome_options)

## 사이트 접속
url = 'https://professor.knou.ac.kr/jaehwachung/index.do'
driver.get(url)

## 사이트 접속 대기
time.sleep(2)

## 페이지 제목 출력
page_source = driver.page_source
tree = html.fromstring(page_source)

title_text = tree.xpath('//title/text()')
print(title_text)

## 드라이버 종료
driver.quit()

['컴퓨터과학과 정재화 교수']



# 실습 시나리오

## 공공데이터 포털 가입 및 데이터 신청

- [https://www.data.go.kr](https://www.data.go.kr)
- 한국환경공단 에어코리아 대기오염정보 데이터 신청

In [11]:
import requests

## 데이터 수집 url 및 api key 설정
url = 'https://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getCtprvnRltmMesureDnsty'
api_key = '2c55f1e2772f2e2e99e9908adb6d32645597e267da82d0a6f54f263783c55137'

params = {
    'serviceKey': api_key,
    'returnType': 'json',
    'numOfRows': '100',
    'pageNo': '1',
    'sidoName': '서울',
    'ver': '1.0'
}

## 데이터 수집
response = requests.get(url, params=params)

## 호출 성공/실패 출력
print(response.json())



{'response': {'body': {'totalCount': 40, 'items': [{'so2Grade': '1', 'coFlag': None, 'khaiValue': '86', 'so2Value': '0.003', 'coValue': '0.3', 'pm25Flag': None, 'pm10Flag': None, 'o3Grade': '2', 'pm10Value': '24', 'khaiGrade': '2', 'pm25Value': '11', 'sidoName': '서울', 'no2Flag': None, 'no2Grade': '1', 'o3Flag': None, 'pm25Grade': '2', 'so2Flag': None, 'dataTime': '2026-10-09 16:00', 'coGrade': '1', 'no2Value': '0.006', 'stationName': '중구', 'pm10Grade': '2', 'o3Value': '0.072'}, {'so2Grade': '1', 'coFlag': None, 'khaiValue': '78', 'so2Value': '0.002', 'coValue': '0.4', 'pm25Flag': None, 'pm10Flag': None, 'o3Grade': '2', 'pm10Value': '27', 'khaiGrade': '2', 'pm25Value': '10', 'sidoName': '서울', 'no2Flag': None, 'no2Grade': '1', 'o3Flag': None, 'pm25Grade': '2', 'so2Flag': None, 'dataTime': '2026-10-09 16:00', 'coGrade': '1', 'no2Value': '0.019', 'stationName': '한강대로', 'pm10Grade': '2', 'o3Value': '0.063'}, {'so2Grade': '1', 'coFlag': None, 'khaiValue': '77', 'so2Value': '0.003', 'coValue'

#중간과제 실습

- [https://www.data.go.kr](https://www.data.go.kr)
- 한국농수산식품유통공사_일별 도,소매 가격정보 조회 데이터 신청

In [24]:
import requests

## 데이터 수집 url 및 api key 설정
url = 'https://apis.data.go.kr/B552845/perDay/price'
api_key = '2c55f1e2772f2e2e99e9908adb6d32645597e267da82d0a6f54f263783c55137'
data = {}

#반복용 변수 선언
years = [str(year) for year in range(2015, 2025)]
variety_codes = ['01', '02', '03', '06']

for year in years:
  for vrty_cd in variety_codes:

    params = {
        'serviceKey': api_key,
        'pageNo': '1',
        'numOfRows': '200',
        'returnType': 'JSON',
        'cond[exmn_ymd::GTE]': year+'0101',
        'cond[exmn_ymd::LTE]': year+'1231',
        'cond[se_cd::EQ]': '02',
        'cond[ctgry_cd::EQ]': '200',
        'cond[item_cd::EQ]': '211',
        'cond[grd_cd::EQ]': '04',
        'cond[sgg_cd::EQ]': '1101',
        'cond[mrkt_cd::EQ]': '0110211',
        'cond[vrty_cd::EQ]': vrty_cd
    }

    ## 데이터 수집
    response = requests.get(url, params=params)

    ## 데이터 병합
    data.update(response.json())

## 호출 성공/실패 출력
print(data)


{'response': {'body': {'dataType': 'JSON', 'items': {'item': [{'ctgry_cd': '200', 'ctgry_nm': '채소류', 'exmn_dd_cnvs_prc': '8400', 'exmn_dd_prc': '8400', 'exmn_ymd': '20240102', 'grd_cd': '04', 'grd_nm': '상품', 'item_cd': '211', 'item_nm': '배추', 'mrkt_cd': '0110211', 'mrkt_nm': '가락도매', 'orgnl_reg_dt': '2024-01-02T15:35:35Z', 'se_cd': '02', 'se_nm': '중도매', 'sgg_cd': '1101', 'sgg_nm': '서울', 'unit': 'kg(그물망 3포기)', 'unit_sz': '10', 'vrty_cd': '06', 'vrty_nm': '월동'}, {'ctgry_cd': '200', 'ctgry_nm': '채소류', 'exmn_dd_cnvs_prc': '8600', 'exmn_dd_prc': '8600', 'exmn_ymd': '20240103', 'grd_cd': '04', 'grd_nm': '상품', 'item_cd': '211', 'item_nm': '배추', 'mrkt_cd': '0110211', 'mrkt_nm': '가락도매', 'orgnl_reg_dt': '2024-01-03T15:44:53Z', 'se_cd': '02', 'se_nm': '중도매', 'sgg_cd': '1101', 'sgg_nm': '서울', 'unit': 'kg(그물망 3포기)', 'unit_sz': '10', 'vrty_cd': '06', 'vrty_nm': '월동'}, {'ctgry_cd': '200', 'ctgry_nm': '채소류', 'exmn_dd_cnvs_prc': '8400', 'exmn_dd_prc': '8400', 'exmn_ymd': '20240104', 'grd_cd': '04', 'grd

In [16]:
years = [str(year) for year in range(2015, 2025)]
variety_code = ['01', '02', '03', '06']
print(years, variety_code)

['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024'] ['01', '02', '03', '06']
